Анализ сырых данных (описываем корпус):
- формат корпуса
- формат пользовательского отзыва (длина, язык)
- соотношение оценок пользователей

In [1]:
import pandas as pd # импортируем библиотеку pandas

df = pd.read_csv('../data/chatgpt_reviews_raw.csv') # загружаем CSV-файл в DataFrame

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              50000 non-null  object
 1   userName              50000 non-null  object
 2   userImage             50000 non-null  object
 3   content               50000 non-null  object
 4   score                 50000 non-null  int64 
 5   thumbsUpCount         50000 non-null  int64 
 6   reviewCreatedVersion  46466 non-null  object
 7   at                    50000 non-null  object
 8   replyContent          1 non-null      object
 9   repliedAt             1 non-null      object
 10  appVersion            46466 non-null  object
dtypes: int64(2), object(9)
memory usage: 4.2+ MB


In [4]:
df['content'].unique

<bound method Series.unique of 0                      my one and only friend tryue friend
1                                                blaa blaa
2                                                  so nice
3                         ዓለም አቀፍ ደረጃ እውቅና ያለው ትልቁ ልዩነት ነው
4                                                 excelent
                               ...                        
49995                                    Very good quality
49996                                                great
49997                                                super
49998    please subscribe my channel my channel name Bl...
49999                                      C'est parfait 👍
Name: content, Length: 50000, dtype: object>

In [5]:
len(df)

50000

In [7]:
df['score'].value_counts()

score
5    37949
4     4997
1     3882
3     2153
2     1019
Name: count, dtype: int64

In [9]:
df['content_len_chars'] = (
    df['content']
    .astype(str)
    .str.len()
)

df['content_len_chars'].describe()

count    50000.000000
mean        30.938080
std         62.336954
min          1.000000
25%          5.000000
50%         11.000000
75%         27.000000
max        500.000000
Name: content_len_chars, dtype: float64

In [13]:
df['content'].head(10)

0                  my one and only friend tryue friend
1                                            blaa blaa
2                                              so nice
3                     ዓለም አቀፍ ደረጃ እውቅና ያለው ትልቁ ልዩነት ነው
4                                             excelent
5    it is an absolutely stunning and wonderful thi...
6                                      good experience
7                                            excellent
8    this app is the best, I got anything I wanted ...
9                                                 good
Name: content, dtype: object

Посмотрим какие еще присутсвуют языки кроме английского при помощи библиотеки langdetect
Вывод: Данная библиотека не очень хорошо отрабатывает на коротких текстах.

In [14]:
# pip install langdetect

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# чтобы результаты были воспроизводимыми
DetectorFactory.seed = 42


def detect_language(text):
    try:
        text = str(text).strip()

        if not text:
            return 'unknown'

        return detect(text)

    except LangDetectException:
        return 'unknown'


# определяем язык
df['language'] = df['content'].apply(detect_language)

# посмотрим распределение
print(df['language'].value_counts())

# примеры
df[['content', 'language']].head()

language
en         20845
so          7628
af          3951
pl          2583
ro          1839
unknown     1453
ca          1291
it          1138
id           849
cs           765
fr           717
nl           636
no           563
sk           558
sw           480
de           476
bn           445
sl           434
tl           430
cy           428
et           293
da           282
es           185
ar           184
hi           181
hr           173
sq           144
fi           133
sv           123
pt            94
tr            89
hu            86
ur            72
vi            69
lv            61
mr            50
fa            49
ta            45
ne            33
lt            28
gu            26
te            23
ru            19
ml            12
kn            11
zh-cn          9
pa             4
ko             3
bg             3
mk             3
ja             2
uk             1
he             1
Name: count, dtype: int64


,content,language
0,my one and only friend tryue friend,en
1,blaa blaa,so
2,so nice,it
3,ዓለም አቀፍ ደረጃ እውቅና ያለው ትልቁ ልዩነት ነው,unknown
4,excelent,pt


In [15]:
df[['content', 'language']].head(40)

,content,language
0,my one and only friend tryue friend,en
1,blaa blaa,so
2,so nice,it
3,ዓለም አቀፍ ደረጃ እውቅና ያለው ትልቁ ልዩነት ነው,unknown
4,excelent,pt
5,it is an absolutely stunning and wonderful thi...,en
6,good experience,es
7,excellent,ca
8,"this app is the best, I got anything I wanted ...",en
9,good,so


Обработаем корпус:
- добавим метки positive / negative
- уберем стоп слова
- разобьем текст на токены
- сбалансируем данные (сейчас преобладают позитивные оценки)

In [17]:
df['label'] = df['score'].apply(lambda x: 'negative' if x in [1,2] else 'positive') # создаём столбец 'label': 1–2 → negative, иначе → positive

In [18]:
df['length'] = df['content'].str.len() # создаём столбец с длиной текста в 'content'
df.groupby('label')['length'].mean() # считаем среднюю длину текста для каждой категории 'label'

label
negative    75.745358
positive    26.068782
Name: length, dtype: float64

In [23]:
from nltk.corpus import stopwords # импортируем стоп-слова
from nltk.tokenize import wordpunct_tokenize # импортируем токенизатор

stop_words = set(stopwords.words('english')) # создаём набор английских стоп-слов

def preprocess(text):
    tokens = wordpunct_tokenize(text.lower()) # приводим к нижнему регистру и разбиваем на токены
    tokens = [w for w in tokens if w.isalpha()] # оставляем только слова (без цифр и символов)
    tokens = [w for w in tokens if w not in stop_words] # убираем стоп-слова
    return tokens

df['tokens'] = df['content'].apply(preprocess) # применяем функцию к столбцу 'content' и сохраняем результат

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              50000 non-null  object
 1   userName              50000 non-null  object
 2   userImage             50000 non-null  object
 3   content               50000 non-null  object
 4   score                 50000 non-null  int64 
 5   thumbsUpCount         50000 non-null  int64 
 6   reviewCreatedVersion  46466 non-null  object
 7   at                    50000 non-null  object
 8   replyContent          1 non-null      object
 9   repliedAt             1 non-null      object
 10  appVersion            46466 non-null  object
 11  content_len_chars     50000 non-null  int64 
 12  language              50000 non-null  object
 13  label                 50000 non-null  object
 14  length                50000 non-null  int64 
 15  tokens                50000 non-null

In [25]:
!pip install spacy

import spacy

nlp = spacy.load("en_core_web_sm")

def preprocess_spacy(text):

    doc = nlp(str(text).lower())

    tokens = [
        token.lemma_
        for token in doc
        if token.is_alpha
        and not token.is_stop
    ]

    return tokens


df['lemmas'] = df['content'].apply(preprocess_spacy)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              50000 non-null  object
 1   userName              50000 non-null  object
 2   userImage             50000 non-null  object
 3   content               50000 non-null  object
 4   score                 50000 non-null  int64 
 5   thumbsUpCount         50000 non-null  int64 
 6   reviewCreatedVersion  46466 non-null  object
 7   at                    50000 non-null  object
 8   replyContent          1 non-null      object
 9   repliedAt             1 non-null      object
 10  appVersion            46466 non-null  object
 11  content_len_chars     50000 non-null  int64 
 12  language              50000 non-null  object
 13  label                 50000 non-null  object
 14  length                50000 non-null  int64 
 15  tokens                50000 non-null

In [27]:
df['lemmas'].head(20)

0                               [friend, tryue, friend]
1                                          [blaa, blaa]
2                                                [nice]
3             [ዓለም, አቀፍ, ደረጃ, እውቅና, ያለው, ትልቁ, ልዩነት, ነው]
4                                            [excelent]
5       [absolutely, stunning, wonderful, thing, study]
6                                    [good, experience]
7                                           [excellent]
8                           [app, good, get, want, app]
9                                                [good]
10                                               [good]
11                                         [nice, tool]
12                                               [good]
13                                       [good, proson]
14                                          [excellent]
15                                               [nice]
16    [nice, app, kirpaya, mujhe, clam, karne, ka, o...
17                                              

In [28]:
df_en = df[df['language'] == 'en'].copy()

In [29]:
df_en.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20845 entries, 0 to 49998
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              20845 non-null  object
 1   userName              20845 non-null  object
 2   userImage             20845 non-null  object
 3   content               20845 non-null  object
 4   score                 20845 non-null  int64 
 5   thumbsUpCount         20845 non-null  int64 
 6   reviewCreatedVersion  19033 non-null  object
 7   at                    20845 non-null  object
 8   replyContent          1 non-null      object
 9   repliedAt             1 non-null      object
 10  appVersion            19033 non-null  object
 11  content_len_chars     20845 non-null  int64 
 12  language              20845 non-null  object
 13  label                 20845 non-null  object
 14  length                20845 non-null  int64 
 15  tokens                20845 non-null  obj

In [33]:
df_en['score'].count()

np.int64(20845)

In [34]:
df_en['score'].value_counts().sort_index()

score
1     2222
2      535
3      971
4     2140
5    14977
Name: count, dtype: int64

In [35]:
negative = df_en[df_en['score'].isin([1, 2])]
positive = df_en[df_en['score'].isin([4, 5])]

positive_sample = positive.sample(
    n=len(negative),
    random_state=42
)

df_balanced = pd.concat([
    negative,
    positive_sample
]).sample(frac=1, random_state=42)

In [37]:
df_balanced['score'].value_counts()

score
5    2423
1    2222
2     535
4     334
Name: count, dtype: int64

In [38]:
df_balanced.to_csv(
    '../data/chatgpt_reviews_clean_balanced_v2.csv',
    index=False
)